## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root, put it on sys.path, then import the shared data loader.
# WHY:  a hard-coded relative path such as ../../datasets/raw/<file>.csv only resolves when
#       the kernel's working directory happens to be this notebook's folder. Open it from the
#       repository root, from JupyterLab at another level, or on Colab where there is no
#       repository at all, and it breaks. load() finds the data wherever you are, downloads
#       it if this machine has none, and says out loud which copy it used.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools.data import load        # load("titanic"), load("creditcard_fraud", prefer="sample"), ...
# -------------------------------------------------------------------------------------
print("Data loader ready - no file paths needed, and this works on Colab too.")

Data loader ready - no file paths needed, and this works on Colab too.


In [2]:
"""
Unit 1 - Exercise 3: Polynomial Regression Practice

Dataset: REAL emergency-call volume by hour of day, from the Montgomery County (PA)
911 dispatch log. Call volume genuinely rises from a pre-dawn low to a late-afternoon
peak, so a straight line cannot fit it — but neither can a degree-10 polynomial without
overfitting the day-to-day noise. That is the bias-variance tradeoff on real data.

Instructions:
1. Load the provided real dataset
2. Create polynomial regression models with different degrees
3. Detect overfitting by comparing train vs test performance
4. Find the optimal polynomial degree
5. Visualize the results
6. Understand the bias-variance tradeoff

Use the provided real dataset. Expect the test MSE to stop improving well before
degree 10: the unexplained part is real randomness in when people call for help,
and no polynomial can fit that.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load a REAL non-linear relationship: hour of day vs number of 911 calls dispatched.
# WHY: nobody wrote a formula for this curve — the county recorded it one call at a time.
# prefer="sample" reads the 25,000-row extract that ships with the repository, so this cell
# runs on any machine and on Colab, and the whole class gets the same 400 points.
calls = load("montgomery_911_calls", prefer="sample", usecols=["timeStamp"])
timestamps = pd.to_datetime(calls["timeStamp"])

# Count the calls that arrived in each clock hour, week by week: one row per (week, hour).
# WHY per week and not per single day: the bundled sample keeps 1 call in every 27, so a
# single (day, hour) slot would hold 0 or 1 call and the daily rhythm would vanish into
# rounding. A week's worth of one clock hour is still a count of real calls.
per_call = pd.DataFrame({"week": timestamps.dt.to_period("W"),
                         "hour_of_day": timestamps.dt.hour})
# The COMPLETE (week x hour) grid, so a week with no calls at 04:00 counts as 0 rather
# than disappearing - dropping those would quietly flatten the quiet hours.
weeks = per_call["week"].drop_duplicates().sort_values()
grid = pd.MultiIndex.from_product([weeks, range(24)], names=["week", "hour_of_day"])
volume = (per_call.groupby(["week", "hour_of_day"]).size()
          .reindex(grid, fill_value=0)
          .reset_index(name="calls"))
volume["hour_of_day"] = volume["hour_of_day"].astype(float)

# Classroom-size it: 400 randomly chosen week-hours (random_state=123 for a shared answer).
sample = volume.sample(n=400, random_state=123).sort_values("hour_of_day")
X = sample[["hour_of_day"]].to_numpy()
y = sample["calls"].to_numpy().astype(float)

df = pd.DataFrame({"x": X.flatten(), "y": y})

print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"x = hour of day (0-23), y = 911 calls dispatched in that hour, in one week")
print(df.head())
print()
print("The real curve you are trying to fit (mean calls per hour of the day, every week):")
print(volume.groupby("hour_of_day")["calls"].mean().round(1).to_string())

# TODO: Write your code here

# Task 1: Split the data into train and test sets (80/20)
print("\n" + "="*60)
print("Task 1: Split data")
print("="*60)
# Your code here...

# Task 2: Train polynomial regression with degree 1 (linear)
print("\n" + "="*60)
print("Task 2: Polynomial Regression (degree=1)")
print("="*60)
# Use PolynomialFeatures with degree=1
# Train LinearRegression
# Evaluate on both train and test
# Your code here...

# Task 3: Train polynomial regression with degree 2
print("\n" + "="*60)
print("Task 3: Polynomial Regression (degree=2)")
print("="*60)
# Use PolynomialFeatures with degree=2
# Train and evaluate
# Your code here...

# Task 4: Train polynomial regression with degree 5
print("\n" + "="*60)
print("Task 4: Polynomial Regression (degree=5)")
print("="*60)
# Use PolynomialFeatures with degree=5
# Train and evaluate
# Notice overfitting (high train accuracy, lower test accuracy)
# Your code here...

# Task 5: Find optimal degree using validation
print("\n" + "="*60)
print("Task 5: Find optimal polynomial degree")
print("="*60)
# Test degrees from 1 to 10
# Plot degree vs MSE (both train and test)
# Find degree with best test performance
# Your code here...

# Task 6: Visualize results
print("\n" + "="*60)
print("Task 6: Visualize polynomial fits")
print("="*60)
# Plot original data
# Plot polynomial fits for different degrees
# Show how higher degrees can overfit
# Your code here...

print("\n" + "="*60)
print("Exercise 3 Complete!")
print("="*60)

montgomery_911_calls: bundled 25,000-row sample of the 663,522-row original (the full file is on this machine but was not used, because prefer='sample') — every number below is for the sample, not the full file. How it was drawn: 1 row in every 27, evenly spread, so the sample covers the same 2015-12-10 to 2020-07-29 window with all 100 call types and 68 townships.
Dataset Info:
Shape: (400, 2)
x = hour of day (0-23), y = 911 calls dispatched in that hour, in one week
     x    y
0  0.0  3.0
1  0.0  3.0
2  0.0  2.0
3  0.0  2.0
4  0.0  3.0

The real curve you are trying to fit (mean calls per hour of the day, every week):
hour_of_day
0.0     2.2
1.0     1.8
2.0     1.7
3.0     1.4
4.0     1.4
5.0     1.8
6.0     2.8
7.0     4.1
8.0     5.1
9.0     5.5
10.0    5.8
11.0    5.7
12.0    6.2
13.0    6.2
14.0    6.3
15.0    6.4
16.0    6.7
17.0    6.9
18.0    5.7
19.0    5.1
20.0    4.4
21.0    3.8
22.0    3.1
23.0    2.7

Task 1: Split data

Task 2: Polynomial Regression (degree=1)

Task 3: 